# 07a · 设计：特征、标签和考试规则

## 这一章在整个项目里的位置

```
00–06 ✅ → [07 特征和基线] → 08 深度学习 → 09 全蛋白组预测 → 10 对外 API
              │
              ├─ 07a 设计：特征清单、标签、考试规则   ← 你在这里
              ├─ 07b 标签：每个实验换算到同一把尺子
              ├─ 07c 结构特征和图：AlphaFold
              ├─ 07d 语言模型特征：ESM-2
              └─ 07e 特征表 + 第一个基线模型 + MLflow
```

## 背景：为什么需要这一章

05 号的 gold 表里已经有了训练数据：94 个实验、80 个人类蛋白、32 万个测量。但模型读不懂 `R175H` 这样的字符串，要先把每个突变翻译成**一串数字**，这串数字就叫**特征**。

旧的 QAFI 代码有 28 个特征，**思路里有好东西，写法很乱**：要在实验室服务器上跑 PSI-BLAST 和 Arpeggio，还有十几张来历说不清的统计表，全蛋白组的算法也没有留下来。所以我们**不搬旧代码，按公司标准重写**。

07a 先把规矩定下来，代码很少：
- 算哪些特征
- 标签是什么
- 怎么考试才不作弊

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 5 步：在你电脑上用旧论文的数据做一个演示
   └─ 第 6 步：把训练/测试的蛋白名单锁住，dbt 把它放进 Databricks 的 gold
看结果：这里的输出，Databricks 网页的 Catalog
```

## 这一章做的 6 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 特征是什么 | 体检报告 |
| 2 | 旧的 28 个特征过一遍 | 留下好思路，换掉烂写法 |
| 3 | 新的特征清单 | 新的体检项目表 |
| 4 | 标签 | 所有实验用同一把尺子 |
| 5 | 考试规则 + 演示 | 考试不能偷看答案 |
| 6 | 锁住考卷 | 考卷封进抽屉，写进 Git |

**花费：0**。**时间**：大约 10 分钟。

---

## 准备

运行下面这一格：定义执行命令的小工具，连上 Databricks。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
import pandas as pd
from databricks.sdk import WorkspaceClient
from qafi_platform import lakehouse
from qafi_platform.lakehouse import sql
from qafi_core import validation
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
GOLD = "qafi_dev.gold"
PAPER = ROOT.parent / "QAFI_CODE_NEW" / "data" / "Dataset_60proteins_features.csv.gz"   # 旧论文的训练集，只当参照答案
print("Databricks:", w.config.host)
print("旧论文数据:", "找到了" if PAPER.exists() else "没找到", PAPER)

---

## 第 1 步 · 特征是什么

### 在干什么
弄清楚"特征"是什么，好特征有什么标准。

### 背景
比喻：**给每个突变开一张体检报告**。报告上有几十项指标：
- 这个位置埋在蛋白里面，还是露在表面
- 这个位置在进化上保不保守
- 换上去的氨基酸，大小、电荷变化多大
- AlphaFold 对这个位置的结构有多确定

模型（07e 和 08 号）就是看着这张报告来打分：这个突变还剩多少功能。

### 公司里什么情况怎么做
公司里判断一个特征好不好，看四条：
1. **能从原始数据重算**：换一个人、换一台机器，算出来一样。旧代码做不到：原料和算法都留在实验室服务器上
2. **训练集和全蛋白组用同一套算法**：否则模型学的东西和用的时候看到的东西不一样，这叫 training-serving skew，是线上模型出错最常见的原因之一
3. **不偷看答案**：特征只能从序列、结构这些"考题"里算，不能用到 DMS 分数
4. **有意义、能解释**：出了问题能查到是哪一项

### 面试怎么说
> "A good feature is reproducible from raw data, computed by the same code for training and for inference, never derived from the labels, and explainable."

---

## 第 2 步 · 旧的 28 个特征过一遍

### 在干什么
逐个看旧特征：哪些思路留下，用标准做法重写；哪些换成标准工具；哪些删掉。

### 背景
旧代码算这 28 个特征，要靠两个外部程序：
- **PatMut**：用 PSI-BLAST 在 UniRef100（一百多 GB 的序列库）里搜同源序列，再用 MUSCLE 比对，算保守性
- **Arpeggio**：在 AlphaFold 结构上算原子之间的接触

另外还要十几张预先统计好的表。这些原料一个都没有留在仓库里。

### 用了什么技术，为什么

| 旧特征（个数） | 在算什么 | 新做法 | 为什么 |
|---|---|---|---|
| BLOSUM62（1） | 这种替换在进化中常不常见 | **保留**，直接查表 | 标准、简单 |
| PSSM、香农熵、序列邻居的熵（3） | 这个位置在同源蛋白里有多保守 | **换成 ESM-2 语言模型**的突变打分和位置熵 | 原来要 PSI-BLAST 搜 UniRef100，全蛋白组没法算；ESM 只要序列 |
| pLDDT、pLDDT 分箱（2） | AlphaFold 对这个位置有多确定 | **保留 pLDDT**，直接读结构文件；分箱删掉 | 分箱这种切法交给模型自己学 |
| colasi（1） | 原子接触多不多，也就是埋得多深 | **换成标准的溶剂可及性（RSA）和接触数** | 原来依赖 Arpeggio 和一张自制的"最大值表" |
| 3D 邻居保守性：fraction、fanc、fbnc（3） | 空间上的邻居保不保守 | 基线里用**一个简单版**：空间邻居的平均保守性；08 号交给 **GNN 自己学** | 这正是图神经网络擅长的事 |
| M.J. potential、NCE-NR（2） | 换氨基酸后，和邻居的"接触能量"变化 | **保留思路，重写成一个简单函数** | 有物理意义；原代码散在几千行里 |
| access. dependent vol.（1） | 体积变化乘以埋藏程度 | **改成 Δ体积 × RSA** | 用标准量组合 |
| neco ×4、lpar、laar（6） | 从统计表里查出来的"兼容性" | **删掉** | 表是从旧数据统计的，复现不了；语言模型和模型自己能学到 |
| PDFF ×9 | 对其他特征做的分布变换 | **删掉** | 树模型和神经网络自己会学这种非线性；而且部署用的表和论文写的方法不一致 |

还有一个藏得很深的问题：旧代码默认"实验里的位置编号 = UniProt 编号 = AlphaFold 编号"，对不上的时候，特征会**悄悄变成空值或 0**。05 号发现 16% 的位置对不上，并且已经换算好了。新代码用的是 05 号换算后的 `uniprot_position`。

### 面试怎么说
> "I audited the 28 legacy features: kept the ideas worth keeping, replaced bespoke pipelines (PSI-BLAST against UniRef100, Arpeggio, KDE lookup tables) with standard reproducible computations, and dropped lookup tables nobody could rebuild."

---

## 第 3 步 · 新的特征清单

### 在干什么
定下新的"体检项目表"，以及每一项在哪一步算。

### 用了什么技术，为什么

| 类别 | 特征 | 从哪里来 | 哪一步算 |
|---|---|---|---|
| 理化 | BLOSUM62、Δ疏水性、Δ体积、Δ电荷 | 突变本身（查表） | 07e |
| 结构 | pLDDT、溶剂可及性 RSA、接触数、二级结构、Δ接触能量 | AlphaFold 结构（raw 里已有） | 07c |
| 语言模型 | ESM-2 突变打分、位置熵 | 蛋白序列（UniProt） | 07d |
| 邻居 | 空间邻居的平均保守性 | 结构 + 语言模型 | 07e |
| 给深度学习用 | 每个位置的 ESM 向量；蛋白的"图" | 07c、07d 存好 | 08 号用 |

**ESM-2** 是 Meta 开源的蛋白质语言模型，读过几亿条蛋白序列，学会了"哪种氨基酸放在哪里合理"。我们先**拿来就用**（07d）：它的参数不动，只让它给每个突变打分、给每个位置一串数字。08 号再学怎么**微调**它。

**蛋白的图**：每个氨基酸是一个点，空间上挨得近的连一条线。08 号的图神经网络（GNN）就在这张图上学习。

### 公司里什么情况怎么做
先用**简单、可解释的特征 + 树模型**做一个基线（07e），再上深度学习（08 号）。深度学习必须比基线好，才说明值得用。

### 面试怎么说
> "Features come in three families from raw data only: physico-chemical change, AlphaFold structure, and ESM-2 language-model scores; per-residue embeddings and residue graphs are stored for the deep-learning models."

---

## 第 4 步 · 标签

### 在干什么
定下模型要学的"答案"是什么。

### 背景
每个 DMS 实验的分数尺子都不一样：有的测细胞生长，有的测荧光，有的测结合；数值范围也各不相同。要放在一起训练，先得**换算到同一把尺子上**。

旧 QAFI 用的是 **Gray 标准化**，这个思路我们保留：
- **1** = 和野生型一样
- **0** = 和最坏的那一批突变（最差的 1%）一样，基本失活
- 大于 1 = 可能功能增强

### 公司里什么情况怎么做
问题是：ProteinGym 没有给每个实验的**野生型分数**，旧代码是一个个手工查的。手工查的数复现不了。07b 会用**数据自己估计**野生型水平，并和旧代码手工查过的那 30 个实验对照，误差小才采用。

### 面试怎么说
> "Each assay is rescaled so 1 means wild-type-like and 0 means as damaging as the worst 1%, so assays with different readouts can be pooled; the wild-type level is estimated from the data, not typed in by hand."

---

## 第 5 步 · 考试规则：不能偷看答案

### 在干什么
定下模型怎么考试，用一个演示看看作弊的分数有多虚。

### 背景
我们的模型是给**没有任何 DMS 数据的蛋白**用的。所以考试必须考它**没见过的蛋白**，三条规矩：
1. **整个蛋白一起留出来**：一个蛋白的所有突变，要么全在训练里，要么全在考试里。不能随机抽突变
2. **近亲一起留出来**：比如 HRAS 和 KRAS，88% 的氨基酸一样。一个在训练里、一个在考试里，就等于提前看了答案
3. **最后的考卷只用一次**：测试蛋白选好就锁住，调参数只用训练蛋白

**旧代码在这里出过问题**：论文的 Test30 测试，训练时用了全部 60 个蛋白，其中就包括那 30 个"没见过"的测试蛋白。

### 演示
用旧论文的训练集（60 个蛋白、29 万个突变、28 个旧特征），同一个模型考两次：一次随机抽突变来考，一次整个蛋白留出来考。只当演示和参照，不用来训练新模型。

In [ ]:
paper = pd.read_csv(PAPER)
OLD_FEATURES = ["Blosum62", "PSSM", "Shannon's entropy", "Shannon's entropy of seq. neighbours", "pLDDT", "pLDDT bin",
                "colasi", "fraction cons. 3D neighbor", "fanc", "fbnc", "M.J. potential", "access.dependent vol.", "neco",
                "PSSM_pdff", "Shannon's entropy_pdff", "Shannon's entropy of seq. neighbours_pdff", "pLDDT_pdff", "colasi_pdff",
                "fraction cons. 3D neighbor_pdff", "fanc_pdff", "fbnc_pdff", "e_native_pdff", "neco2", "neco3", "neco12",
                "lpar", "NCE-NR", "laar"]
validation.compare_splits(paper, OLD_FEATURES, "score_log_normalized", "uniprot")

### 结果怎么看
- **随机抽突变**：中位 Pearson 约 **0.485**
- **整个蛋白留出来**：中位 Pearson 约 **0.43**

同一个模型、同一份数据，只是考法不同，成绩就虚高了 0.06。原因是随机抽的时候，模型在训练里已经见过同一个蛋白、甚至同一个位置的其他突变。**只有第二个数，才是它对一个新蛋白的真实水平。**

### 公司里什么情况怎么做
- 先想清楚"模型将来会遇到什么"，考试就照那个样子考。我们的模型将来遇到的是新蛋白，所以按蛋白留出
- 交叉验证用分组的版本（GroupKFold），组就是"一家近亲"
- 成绩按蛋白分别算，再取中位数，和 ProteinGym、旧论文的口径一样

### 面试怎么说
> "Splitting variants at random inflated the median per-protein correlation from 0.43 to 0.49, because the model had seen other variants of the same protein. The model is meant for unseen proteins, so I validate with whole proteins held out, grouping homologs together."

---

## 第 6 步 · 锁住考卷

### 在干什么
从 80 个有 DMS 数据的蛋白里，选出最后考试用的蛋白，写成一个文件，提交进 Git。

### 背景
比喻：**考卷封进抽屉**。现在就选好，以后不管模型怎么改，都不许看这些蛋白的成绩来调参数；只在最后考一次。

### 用了什么技术，为什么
1. **找近亲**：两两比对 80 个蛋白的序列（BLAST 用的那套统计，看 E 值），满足三条才算近亲：不像是巧合（E 值 < 1e-5）、至少 30% 一样、覆盖较短蛋白的一半以上。这也是 MMseqs2、CD-HIT 这类聚类工具常用的标准
2. **按"家"分**：近亲是一家，一家人整个去训练或者整个去考试。按家名算一个固定的哈希值，大约 20% 的家去考试。用哈希而不是随机抽：以后加了新蛋白，原来的分配也不会变
3. **写成 dbt 的 seed**：seed 就是放在 Git 里的一个小表格文件，dbt 把它装进 gold。谁改了它，Git 里都有记录

### 执行
第一次运行时生成 `dbt/seeds/protein_split.csv`；文件已经存在，就只读不改：

In [ ]:
proteins = sql(w, f"SELECT uniprot_id, gene_name, sequence FROM {GOLD}.dim_protein WHERE has_dms")
split = validation.lock_split(ROOT / "dbt" / "seeds" / "protein_split.csv", dict(zip(proteins.uniprot_id, proteins.sequence)))
split = split.merge(proteins[["uniprot_id", "gene_name"]])
print("近亲：", [sorted(g.gene_name) for _, g in split.groupby("similarity_group") if len(g) > 1])
print("测试蛋白：", sorted(split[split.split == "test"].gene_name))
split.split.value_counts()

### 结果怎么看
- **近亲**：3 家，都是真正的同源蛋白。HRAS 和 KRAS；CASP3 和 CASP7；还有 6 个共用激酶、SH2、SH3 结构域的蛋白（SRC、MAPK1、EPHB2、GRB2、OBSCN、SORBS1）
- **测试蛋白**：15 个，里面有 TP53、BRCA1、BRCA2；HRAS 和 KRAS 一起去了测试
- **train 65，test 15**

把这个表装进 Databricks 的 gold，并检查三件事：每个有 DMS 的蛋白都分了边、近亲在同一边、只写了 train 或 test：

In [ ]:
os.environ.update(lakehouse.dbt_env(w))
run("cd dbt && uv run dbt build --select protein_split+ --profiles-dir .")

### 结果怎么看
最后一行 `Done. PASS=9 WARN=0 ERROR=0`：装好了表，8 项检查全部通过。

训练和考试各有多少数据：

In [ ]:
sql(w, f'''
SELECT s.split, count(DISTINCT s.uniprot_id) AS proteins, count(DISTINCT m.dms_id) AS assays, count(*) AS measurements
FROM {GOLD}.protein_split s JOIN {GOLD}.fct_dms_measurement m ON m.uniprot_id = s.uniprot_id
GROUP BY s.split ORDER BY s.split DESC''')

### 结果怎么看
- **train**：65 个蛋白、74 个实验、约 25.2 万个测量
- **test**：15 个蛋白、20 个实验、约 7.1 万个测量，大约 22%

### 公司里什么情况怎么做
- 测试集在**第一次训练之前**就定好，写进代码库
- 以后来了新的 DMS 蛋白（比如 ProteinGym 出新版），dbt 的检查会失败，提醒你给新蛋白分边；原来的蛋白不动

### 面试怎么说
> "The test proteins were chosen before any model existed, by a fixed hash over homology groups, and committed as a dbt seed with tests that every DMS protein has a side and that homologs never straddle the split."

---

## 提交代码

这次提交也包括 06 号的一个小修复：查 Airflow 运行状态的小工具，改成了通过 Airflow 的网页接口来查。

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 07a validation design and locked test proteins") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 决定了什么 | 为什么 |
|---|---|---|
| 1 特征 | 好特征的四条标准 | 可复现、训练和使用一致、不偷看答案、能解释 |
| 2 旧特征 | 留下好思路，换掉 PSI-BLAST、Arpeggio 和统计表 | 旧原料复现不了，全蛋白组算不了 |
| 3 新特征 | 理化 + 结构 + 语言模型，外加给深度学习的向量和图 | 都能从 raw 算，全蛋白组也能算 |
| 4 标签 | Gray 标准化，野生型水平从数据估计 | 不同实验放在同一把尺子上 |
| 5 考试 | 整个蛋白留出来，近亲一起 | 随机切分会让成绩虚高 |
| 6 考卷 | 15 个测试蛋白，写进 Git | 先锁住，最后只考一次 |

**一句话**：动手算特征之前，先定好考试规则，把考卷锁进抽屉。这样后面每个模型的成绩都可信，而且彼此可比。

## 面试题

1. **How do you validate a model meant for proteins without data?**
   > "Hold out whole proteins and group homologs, so the test mimics a protein the model has never seen. A random split of variants inflated our median correlation from 0.43 to 0.49."

2. **What is training-serving skew?**
   > "Features computed one way for training and another way in production. The model then sees different inputs from the ones it learned on. One code path for both prevents it."

3. **How do you keep a test set honest over months of work?**
   > "Choose it before the first model, make the choice deterministic, commit it, and test that it never changes; tune only with cross-validation on the training proteins."

**下一步 07b · 标签**：把 94 个实验的分数换算到同一把尺子上，并和旧代码手工查的值对照。